# Arkham Data Science Challenge

## Task 1: Data extraction

In [18]:
import requests as rq
import pandas as pd
import openpyxl
import time
from datetime import date, timedelta

In [4]:
nodos = pd.read_excel('catalogonodos.xlsx', header = 1)
nodos.columns

Index(['SISTEMA', 'CENTRO DE CONTROL REGIONAL', 'ZONA DE CARGA', 'CLAVE',
       'NOMBRE', 'NIVEL DE TENSIÓN (kV)', 'DIRECTAMENTE MODELADA',
       'INDIRECTAMENTE MODELADA', 'DIRECTAMENTE MODELADA.1',
       'INDIRECTAMENTE MODELADA.1', 'ZONA DE OPERACIÓN DE TRANSMISIÓN',
       'GERENCIA REGIONAL DE TRANSMISIÓN', 'ZONA DE DISTRIBUCIÓN',
       'GERENCIA DIVISIONAL DE DISTRIBUCIÓN',
       'CLAVE DE ENTIDAD FEDERATIVA (INEGI)', 'ENTIDAD FEDERATIVA (INEGI)',
       'CLAVE DE MUNICIPIO (INEGI)', 'MUNICIPIO (INEGI)',
       'REGION DE TRANSMISION'],
      dtype='str')

In [5]:
len(nodos)

2585

In [6]:
#True if ('Monclova' in nodos['ZONA DE CARGA'].unique()) else False
nodos['ZONA DE CARGA'].unique()

<StringArray>
[      'ENSENADA',       'MEXICALI',        'SANLUIS',        'TIJUANA',
      'No Aplica',   'CONSTITUCION',         'LA PAZ',      'LOS CABOS',
 'CENTRO ORIENTE',     'CENTRO SUR',
 ...
         'XALAPA',    'ZIHUATANEJO',       'CAMPECHE',         'CANCUN',
         'CARMEN',       'CHETUMAL',         'MERIDA',  'MOTUL TIZIMIN',
   'RIVIERA MAYA',          'TICUL']
Length: 109, dtype: str

In [7]:
nodos_mclv = nodos[nodos['ZONA DE CARGA'].str.lower() == 'monclova']
nodos_mclv_str = ','.join(nodos_mclv['CLAVE'])
nodos_mclv_str

'06AEO-115,06AHM-400,06FRO-115,06GRA-115,06MON-115,06MON-230,06RGS-115,06TKS-115,06XOC-115'

In [ ]:
sistema = 'SIN'
proceso = 'MDA'
lista_nodos = nodos_mclv_str
anio_ini = '2024'
mes_ini = '01'
dia_ini = '01'
anio_fin = '2024'
mes_fin = '01'
dia_fin = '02'
formato = 'JSON'

url = f'https://ws01.cenace.gob.mx:8082/SWPML/SIM/{sistema}/{proceso}/{lista_nodos}/{anio_ini}/{mes_ini}/{dia_ini}/{anio_fin}/{mes_fin}/{dia_fin}/{formato}'
respuesta = rq.get(url)

if respuesta.status_code == 200:
    
    # 3. Conviertes la respuesta directamente a un diccionario de Python
    datos_json = respuesta.json()
    
    # Aquí ya puedes acceder a datos_json['Resultados'] y procesarlos
    print(f"DAtos para el request a {url}")
    # 1. Load the in-memory JSON data directly
    df = pd.DataFrame(datos_json)

    # Save as CSV
    #df.to_csv("resultado.csv", index=False)
    df = df['Resultados']
    print(df)
else:
    print(f"Ups, error en el servidor: {respuesta.status_code}")

# 4. Haces una pequeña pausa antes de la siguiente vuelta del bucle
time.sleep(2)

DAtos para el request a https://ws01.cenace.gob.mx:8082/SWPML/SIM/SIN/MDA/06AEO-115,06AHM-400,06FRO-115,06GRA-115,06MON-115,06MON-230,06RGS-115,06TKS-115,06XOC-115/2024/01/01/2024/01/02/JSON
0    {'clv_nodo': '06AEO-115', 'Valores': [{'fecha'...
1    {'clv_nodo': '06AHM-400', 'Valores': [{'fecha'...
2    {'clv_nodo': '06FRO-115', 'Valores': [{'fecha'...
3    {'clv_nodo': '06GRA-115', 'Valores': [{'fecha'...
4    {'clv_nodo': '06MON-115', 'Valores': [{'fecha'...
5    {'clv_nodo': '06MON-230', 'Valores': [{'fecha'...
6    {'clv_nodo': '06RGS-115', 'Valores': [{'fecha'...
7    {'clv_nodo': '06TKS-115', 'Valores': [{'fecha'...
8    {'clv_nodo': '06XOC-115', 'Valores': [{'fecha'...
Name: Resultados, dtype: object


In [ ]:
def flattenData(form):
    flattenedRows = []
    for nodo in form['Resultados']:
        clave = nodo['clv_nodo']
        for valores in nodo['Valores']:
            valores['nodo'] = clave 
            flattenedRows.append(valores)

    return pd.DataFrame(flattenedRows)

In [10]:
flattenData(datos_json)

,fecha,hora,pml,pml_ene,pml_per,pml_cng,nodo
0,2024-01-01,1,279.92,370.71,-43.69,-47.1,06AEO-115
1,2024-01-01,2,279.21,311.75,-32.54,0,06AEO-115
2,2024-01-01,3,274.23,305.52,-31.28,0,06AEO-115
3,2024-01-01,4,264.01,294.23,-30.22,0,06AEO-115
4,2024-01-01,5,243.44,267.66,-24.22,0,06AEO-115
...,...,...,...,...,...,...,...
427,2024-01-02,20,594.32,649.19,-54.86,0,06XOC-115
428,2024-01-02,21,392.26,431.03,-38.77,0,06XOC-115
429,2024-01-02,22,592.27,646.27,-52.38,-1.62,06XOC-115
430,2024-01-02,23,490.33,535.61,-45.28,0,06XOC-115


In [19]:
def getRequestURLs(startDate, endDate, lista_nodos, sistema='SIN', proceso='MDA', formato='JSON'):
    URLlist = []
    current_date = startDate
    
    while current_date <= endDate: 
    
        chunk_end = current_date + timedelta(days=6)
        if chunk_end > endDate:
            chunk_end = endDate

        # .strftime("%Y/%m/%d") asegura el formato exacto que pide el CENACE
        f_ini = current_date.strftime("%Y/%m/%d")
        f_fin = chunk_end.strftime("%Y/%m/%d")
        
        url = f"https://ws01.cenace.gob.mx:8082/SWPML/SIM/{sistema}/{proceso}/{lista_nodos}/{f_ini}/{f_fin}/{formato}"

        URLlist.append(url)

        current_date = chunk_end + timedelta(days=1)
        
    return URLlist

In [23]:
urlList = getRequestURLs(date(2024,1,1), date(2025,6,30), nodos_mclv_str)
urlList

['https://ws01.cenace.gob.mx:8082/SWPML/SIM/SIN/MDA/06AEO-115,06AHM-400,06FRO-115,06GRA-115,06MON-115,06MON-230,06RGS-115,06TKS-115,06XOC-115/2024/01/01/2024/01/07/JSON',
 'https://ws01.cenace.gob.mx:8082/SWPML/SIM/SIN/MDA/06AEO-115,06AHM-400,06FRO-115,06GRA-115,06MON-115,06MON-230,06RGS-115,06TKS-115,06XOC-115/2024/01/08/2024/01/14/JSON',
 'https://ws01.cenace.gob.mx:8082/SWPML/SIM/SIN/MDA/06AEO-115,06AHM-400,06FRO-115,06GRA-115,06MON-115,06MON-230,06RGS-115,06TKS-115,06XOC-115/2024/01/15/2024/01/21/JSON',
 'https://ws01.cenace.gob.mx:8082/SWPML/SIM/SIN/MDA/06AEO-115,06AHM-400,06FRO-115,06GRA-115,06MON-115,06MON-230,06RGS-115,06TKS-115,06XOC-115/2024/01/22/2024/01/28/JSON',
 'https://ws01.cenace.gob.mx:8082/SWPML/SIM/SIN/MDA/06AEO-115,06AHM-400,06FRO-115,06GRA-115,06MON-115,06MON-230,06RGS-115,06TKS-115,06XOC-115/2024/01/29/2024/02/04/JSON',
 'https://ws01.cenace.gob.mx:8082/SWPML/SIM/SIN/MDA/06AEO-115,06AHM-400,06FRO-115,06GRA-115,06MON-115,06MON-230,06RGS-115,06TKS-115,06XOC-115/202

In [24]:
lista_chunks = []
for link in urlList:
    respuesta = rq.get(link)

    if respuesta.status_code == 200:
        rowsChunk = flattenData(respuesta.json())
        print(f"Datos para el request a {link[-30:]}")
        print(rowsChunk)
        lista_chunks.append(rowsChunk)
        
    else:
        print(f"Ups, error en el servidor: {respuesta.status_code}")

    time.sleep(2)

Datos para el request a 115/2024/01/01/2024/01/07/JSON
           fecha hora     pml pml_ene pml_per pml_cng       nodo
0     2024-01-01    1  279.92  370.71  -43.69   -47.1  06AEO-115
1     2024-01-01    2  279.21  311.75  -32.54       0  06AEO-115
2     2024-01-01    3  274.23  305.52  -31.28       0  06AEO-115
3     2024-01-01    4  264.01  294.23  -30.22       0  06AEO-115
4     2024-01-01    5  243.44  267.66  -24.22       0  06AEO-115
...          ...  ...     ...     ...     ...     ...        ...
1507  2024-01-07   20  483.26  603.07  -76.33  -43.48  06XOC-115
1508  2024-01-07   21  482.72  602.26   -75.5  -44.03  06XOC-115
1509  2024-01-07   22  458.49  581.47  -72.13  -50.85  06XOC-115
1510  2024-01-07   23  410.74  468.46  -57.72       0  06XOC-115
1511  2024-01-07   24  375.39  428.35  -52.95       0  06XOC-115

[1512 rows x 7 columns]
Datos para el request a 115/2024/01/08/2024/01/14/JSON
           fecha hora     pml  pml_ene  pml_per  pml_cng       nodo
0     2024-01-08 

In [ ]:
# dataframe consolidado
info = pd.concat(lista_chunks, ignore_index=True)

info.to_csv('monclova_pml_2024_2025.csv', index=False)

print(f"Filas totales recuperadas: {info.shape[0]}")
print(f"Columnas recuperadas: {info.shape[1]}")
info.head()

Filas totales recuperadas: 118152
Columnas recuperadas: 7


,fecha,hora,pml,pml_ene,pml_per,pml_cng,nodo
0,2024-01-01,1,279.92,370.71,-43.69,-47.1,06AEO-115
1,2024-01-01,2,279.21,311.75,-32.54,0,06AEO-115
2,2024-01-01,3,274.23,305.52,-31.28,0,06AEO-115
3,2024-01-01,4,264.01,294.23,-30.22,0,06AEO-115
4,2024-01-01,5,243.44,267.66,-24.22,0,06AEO-115


## Task 2: Modeling Tracks​